💡 **Environment:** `clamp-analyses`  


# Description

Projects LINCS L1000 consensus drug signatures into the recount2 CLAMP latent space.

**Input**: `data/drug_disease_associations/lincs-data.pkl` (7120 Ensembl genes × 1170 drugs).

**Output**: `05_lincs_projection_recount2/lincs/lincs-projection.pkl` (LVs × drugs).


# Modules loading


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

from pyprojroot import here


# Settings


In [3]:
CLAMP_MODEL_FILE = here('output/01_model_building/03_recount2/01_recount2_hall/CLAMPfull_hall.rds')
display(CLAMP_MODEL_FILE)
assert CLAMP_MODEL_FILE.exists()


PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/01_model_building/03_recount2/01_recount2_hall/CLAMPfull_hall.rds')

In [4]:
# Input: processed LINCS data (Ensembl IDs, 7120 PhenomeXcan genes)
DATA_DIR = here('data/drug_disease_associations')
LINCS_INPUT_FILE = DATA_DIR / 'lincs-data.pkl'
display(LINCS_INPUT_FILE)
assert LINCS_INPUT_FILE.exists()

# Output
OUTPUT_DIR = here('output/03_model_biology/00_archs4/02_drug_disease_associations/05_lincs_projection_recount2') / 'lincs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
display(OUTPUT_DIR)


PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/data/drug_disease_associations/lincs-data.pkl')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/05_lincs_projection_recount2/lincs')

# Projection helpers


In [5]:
def prepare_clamp_projector(clamp_model_file):
    CLAMP = importr('CLAMP')
    readRDS = ro.r['readRDS']
    clamp = readRDS(str(clamp_model_file))
    print('CLAMP model loaded')

    gene_symbols = list(ro.r['rownames'](clamp.rx2('Z')))
    lv_names = list(ro.r['colnames'](clamp.rx2('Z')))
    print(f'CLAMP genes: {len(gene_symbols)}, LVs: {len(lv_names)}')

    # Map CLAMP gene symbols (HGNC) to Ensembl IDs used by LINCS and S-PrediXcan.
    clusterProfiler = importr('clusterProfiler')
    bitr_result = clusterProfiler.bitr(
        ro.StrVector(gene_symbols),
        fromType='SYMBOL',
        toType='ENSEMBL',
        OrgDb='org.Hs.eg.db',
    )

    with localconverter(ro.default_converter + pandas2ri.converter):
        mapping_df = ro.conversion.rpy2py(bitr_result)

    print(f'Raw mapping shape: {mapping_df.shape}')
    display(mapping_df.head())

    # Keep only 1:1 unambiguous symbol <-> Ensembl mappings.
    dup_symbols = mapping_df['SYMBOL'].duplicated(keep=False)
    dup_ensembl = mapping_df['ENSEMBL'].duplicated(keep=False)
    mapping_1to1 = mapping_df[~dup_symbols & ~dup_ensembl].set_index('SYMBOL')
    print(f'1:1 mappings: {mapping_1to1.shape[0]} / {len(gene_symbols)} CLAMP genes')

    mapped_symbols = mapping_1to1.index.tolist()
    mapped_ensembl = mapping_1to1['ENSEMBL'].tolist()

    subset_Z = ro.r('function(clamp, genes) { clamp$Z <- as.matrix(clamp$Z[genes, ]); clamp }')
    clamp_sub = subset_Z(clamp, ro.StrVector(mapped_symbols))
    print(f'Subsetted CLAMP Z: {len(mapped_symbols)} genes x {len(lv_names)} LVs')

    return CLAMP, clamp_sub, mapped_ensembl, lv_names


def project_to_clamp(data, CLAMP, clamp_sub, mapped_ensembl, lv_names):
    aligned = data.reindex(mapped_ensembl).fillna(0.0).values

    r_mat = ro.r['matrix'](
        ro.FloatVector(aligned.flatten('F')),
        nrow=aligned.shape[0],
        ncol=aligned.shape[1],
    )

    proj_r = CLAMP.projectCLAMP(clamp_sub, newdata=r_mat)

    with localconverter(ro.default_converter + pandas2ri.converter):
        proj_values = ro.conversion.rpy2py(proj_r)

    return pd.DataFrame(proj_values, index=lv_names, columns=data.columns)


# Load LINCS data


In [6]:
input_file = LINCS_INPUT_FILE
display(input_file)
lincs_data = pd.read_pickle(input_file)
display(lincs_data.shape)
display(lincs_data.head())
assert lincs_data.index.is_unique
assert lincs_data.columns.is_unique
assert not lincs_data.isna().any().any()


PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/data/drug_disease_associations/lincs-data.pkl')

(7120, 1170)

perturbagen,DB00014,DB00091,DB00121,DB00130,DB00131,DB00132,DB00136,DB00140,DB00146,DB00150,...,DB08995,DB09002,DB09004,DB09009,DB09010,DB09015,DB09019,DB09020,DB09022,DB09023
ENSG00000196839,-1.001,-1.835,1.391,1.132,0.257,1.932,0.508,1.408,0.777,0.032,...,-1.692,-0.516,-1.435,-0.317,-0.012,0.641,-0.230,-0.518,-0.177,2.146
ENSG00000170558,1.146,-1.863,0.011,-1.020,1.143,-0.115,1.327,0.310,-1.853,0.872,...,0.354,0.498,0.268,-1.084,-0.142,-0.077,0.633,-1.807,0.032,0.135
ENSG00000117020,-0.693,1.694,-0.804,-0.164,1.145,-1.465,1.221,-0.747,0.829,-0.961,...,-1.196,-0.230,-1.049,-0.347,0.586,0.865,-0.021,2.180,-0.956,0.105
ENSG00000133997,-0.037,0.383,0.269,-0.997,0.185,-0.536,0.424,-0.119,-1.313,0.579,...,-0.343,0.116,-0.245,-0.127,-1.367,0.149,0.117,2.084,1.178,0.772
ENSG00000101473,0.162,-0.899,0.105,-0.090,-1.291,1.404,0.185,0.157,-0.327,-0.026,...,-0.136,-1.115,-0.280,0.200,0.638,-0.197,-0.360,-2.302,-0.117,-0.167


# Prepare CLAMP projector


In [7]:
CLAMP, clamp_sub, mapped_ensembl, lv_names = prepare_clamp_projector(CLAMP_MODEL_FILE)


CLAMP model loaded
CLAMP genes: 6000, LVs: 724


R callback write-console: 
  


R callback write-console: 'select()' returned 1:many mapping between keys and columns
  


Raw mapping shape: (6715, 2)


,SYMBOL,ENSEMBL
1,GAS6,ENSG00000183087
2,MMP14,ENSG00000157227
3,DSP,ENSG00000096696
4,MARCKSL1,ENSG00000175130
5,SPARC,ENSG00000113140


...

R callback write-console: Warning message:
  


R callback write-console: In (function (geneID, fromType, toType, OrgDb, drop = TRUE)  :  


R callback write-console: 
   


R callback write-console:  0.18% of input gene IDs are fail to map...
  


1:1 mappings: 5604 / 6000 CLAMP genes
Subsetted CLAMP Z: 5604 genes x 724 LVs


# Project LINCS


In [8]:
lincs_projection_file = OUTPUT_DIR / 'lincs-projection.pkl'
display(lincs_projection_file)

lincs_projection = project_to_clamp(
    lincs_data, CLAMP, clamp_sub, mapped_ensembl, lv_names
)
print(f'LINCS projection shape: {lincs_projection.shape}')
assert not lincs_projection.isna().any().any()
lincs_projection.to_pickle(lincs_projection_file)
print('Saved.')

display(lincs_projection.head())


PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/05_lincs_projection_recount2/lincs/lincs-projection.pkl')

LINCS projection shape: (724, 1170)
Saved.


perturbagen,DB00014,DB00091,DB00121,DB00130,DB00131,DB00132,DB00136,DB00140,DB00146,DB00150,...,DB08995,DB09002,DB09004,DB09009,DB09010,DB09015,DB09019,DB09020,DB09022,DB09023
LV1,0.003179,-0.215170,0.060520,-0.096349,0.024118,0.007310,0.086585,-0.004956,-0.042509,-0.010136,...,0.037589,0.079428,-0.024861,-0.025116,-0.070027,0.038477,0.045107,0.128886,0.054431,-0.007958
LV2,0.023899,-0.042962,0.009088,-0.033447,0.009802,-0.011031,0.112337,-0.000358,0.009663,0.009500,...,-0.048660,0.000381,-0.001468,-0.010170,0.013613,-0.029456,-0.059639,-0.012081,0.026457,-0.003287
LV3,-0.008056,0.065900,0.065524,0.023932,-0.040531,0.099507,-0.018339,0.016091,0.022351,-0.015659,...,-0.038714,-0.003951,0.021199,0.030703,-0.083161,-0.000687,0.059978,0.040192,0.042454,-0.000282
LV4,0.001010,-0.014094,0.023886,0.000340,0.009103,0.008644,-0.036792,0.017644,-0.024312,-0.001480,...,0.031384,0.026598,0.005721,0.015177,0.003279,0.001539,0.045908,0.111801,0.030689,0.000260
LV5,-0.012695,0.252748,-0.075861,-0.001037,-0.015189,-0.172504,-0.064784,-0.012453,0.005494,-0.009908,...,-0.004436,-0.021837,-0.034371,-0.043902,0.049067,0.039357,-0.073870,0.105535,-0.049924,0.010845
